# scalper-hft: дослідницький ноутбук

Інтерактивний цикл: дані → фічі → стратегія → бектест → анти-перенавчання.

```bash
.venv/bin/python -m ipykernel_launcher --ipython-dir=.  # або запустити з PyCharm/Jupyter
```

In [ ]:
import logging
import sys

sys.path.insert(0, ".")
logging.basicConfig(level=logging.WARNING)

from scalper_hft.backtest.engine import run_backtest
from scalper_hft.backtest.execution import CostModel
from scalper_hft.config import get_settings
from scalper_hft.data.downloader import download_agg_trades, download_klines
from scalper_hft.strategies import get_strategy
from scalper_hft.validation.deflated_sharpe import deflated_sharpe_ratio, estimate_n_trials
from scalper_hft.validation.walk_forward import run_walk_forward

s = get_settings()
df = download_klines("BTCUSDT", "1m", days=10)
print(f"Дані: {len(df)} барів, {df.index[0]} → {df.index[-1]}")
df.tail(3)

In [ ]:
# Бектест стратегії
strat = get_strategy("mean_reversion", rsi_period=7, oversold=40, overbought=60, bb_period=15, min_atr_pct=0.0005)
cost = CostModel(maker_fee=s.maker_fee, taker_fee=s.taker_fee, slippage_frac=s.slippage_frac)
res = run_backtest(df, strat, cost=cost, position_pct=s.position_pct)
print(res.summary())

In [ ]:
# Аудит перенавчання: walk-forward + deflated Sharpe
wf = run_walk_forward(df, strat, train_bars=3000, test_bars=1000, cost=cost)
print(wf.summary())

ret = res.equity.pct_change().dropna()
dsr = deflated_sharpe_ratio(ret.values, n_trials=estimate_n_trials(len(strat.param_space), 40))
print(f"\nDeflated Sharpe: {dsr:.3f} (edge значущий якщо > 0.95)")

In [ ]:
# Мікроструктура: CVD з aggTrades (потрібен завантажений файл trades)
from scalper_hft.data.storage import trades_path

if trades_path(s.data_dir_abs, "BTCUSDT").exists():
    trades = download_agg_trades("BTCUSDT", days=1)
    from scalper_hft.features.indicators import cvd_from_trades

    cvd = cvd_from_trades(trades, "1min")
    cvd["cvd"].plot(figsize=(12, 4), title="Cumulative Volume Delta (BTCUSDT)")
else:
    print(
        "Спершу: python -m scalper_hft.cli download --symbol BTCUSDT --interval 1m --days 10 --trades --trades-days 1"
    )